# Stability analysis

In [ ]:
import asyncio
import json
import boto3
import pandas as pd

subsample_task_ids = {"106", "145", "137", "1145", "711"}

number_of_resample = 3

EPOCH_TO_MODEL = {
    "1": "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    "2": "us.anthropic.claude-sonnet-4-6",
    "3": "deepseek.v3.2",
    "4": "moonshotai.kimi-k2.5",
    "5": "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    "6": "us.anthropic.claude-sonnet-4-6",
    "7": "deepseek.v3.2",
    "8": "moonshotai.kimi-k2.5",
    "9": "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    "10": "us.anthropic.claude-sonnet-4-6",
    "11": "deepseek.v3.2",
    "12": "moonshotai.kimi-k2.5",
    "13": "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    "14": "us.anthropic.claude-sonnet-4-6",
    "15": "deepseek.v3.2",
    "16": "moonshotai.kimi-k2.5",
}

EPOCH_TO_LABEL = {
    "1": "Claude Sonnet 4.5 v2 prompt",
    "2": "Claude Sonnet 4.6 v2 prompt",
    "3": "DeepSeek R1 v2 prompt",
    "4": "Kimi K2 v2 prompt",
    "5": "Claude Sonnet 4.5 v1 prompt",
    "6": "Claude Sonnet 4.6 v1 prompt",
    "7": "DeepSeek R1 v1 prompt",
    "8": "Kimi K2 v1 prompt",
    "9": "Claude Sonnet 4.5 v3 prompt",
    "10": "Claude Sonnet 4.6 v3 prompt",
    "11": "DeepSeek R1 v3 prompt",
    "12": "Kimi K2 v3 prompt",
    "13": "Claude Sonnet 4.5 v4 prompt",
    "14": "Claude Sonnet 4.6 v4 prompt",
    "15": "DeepSeek R1 v4 prompt",
    "16": "Kimi K2 v4 prompt",
}

bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")

## Running the inference

In [ ]:
MAX_RETRIES = 3


def load_subsample_prompts(epoch: str) -> dict[str, tuple[list[dict], float]]:
    """Load scorer prompts for subsample tasks from batch input. Returns {record_id: (messages, temperature)}."""
    path = f"input/scorer/test-100-claude-haiku/{epoch}/input.jsonl"
    prompts = {}
    with open(path) as f:
        for line in f:
            record = json.loads(line)
            task_id = record["recordId"].split("/")[0]
            if task_id in subsample_task_ids:
                messages = record["modelInput"]["messages"]
                temperature = record["modelInput"]["inferenceConfig"]["temperature"]
                prompts[record["recordId"]] = (messages, temperature)
    return prompts


def parse_judgement(text: str) -> bool | None:
    """Parse true/false from the last line of judge output."""
    if not text:
        return None
    last_line = text.strip().splitlines()[-1].strip().lower()
    if  "true" in last_line:
        return True
    if "false" in last_line:
        return False
    return None


async def call_judge(model_id: str, messages: list[dict], temperature: float) -> tuple[bool | None, str | None]:
    for attempt in range(MAX_RETRIES):
        try:
            response = await asyncio.to_thread(
                bedrock.converse,
                modelId=model_id,
                messages=messages,
                inferenceConfig={"temperature": temperature},
            )
            text = response["output"]["message"]["content"][0]["text"]
            result = parse_judgement(text)
            if result is not None:
                return result, text
        except Exception as e:
            print(f"    Attempt {attempt + 1}/{MAX_RETRIES} failed: {e}")
        await asyncio.sleep(2 ** attempt)
    return None, None

In [ ]:
from pathlib import Path

out_dir = Path("output/stability_analysis")
out_dir.mkdir(parents=True, exist_ok=True)
results_path = out_dir / "all_results.jsonl"


async def resample_criterion(model_id, rid, messages, temperature):
    pairs = await asyncio.gather(*[
        call_judge(model_id, messages, temperature)
        for _ in range(number_of_resample)
    ])
    results = [judgement for judgement, _ in pairs]
    responses = [text for _, text in pairs]
    return rid, results, responses


all_results = {}

for epoch, model_id in EPOCH_TO_MODEL.items():
    label = EPOCH_TO_LABEL[epoch]
    print(f"--- {label} (epoch {epoch}) ---")
    prompts = load_subsample_prompts(epoch)
    print(f"  {len(prompts)} criteria to resample")

    epoch_results = {}
    tasks = [
        resample_criterion(model_id, rid, messages, temperature)
        for rid, (messages, temperature) in prompts.items()
    ]

    for coro in asyncio.as_completed(tasks):
        rid, results, responses = await coro
        epoch_results[rid] = results
        print(f"  {rid}: {results}")
        with open(results_path, "a") as f:
            f.write(json.dumps({"epoch": epoch, "record_id": rid, "results": results, "responses": responses}) + "\n")

    all_results[epoch] = epoch_results
    print()

## Analyzing the results

In [2]:
from pathlib import Path

results_path = Path("output/stability_analysis/all_results.jsonl")

all_results = {}
with open(results_path) as f:
    for line in f:
        record = json.loads(line)
        epoch = record["epoch"]
        if epoch in EPOCH_TO_LABEL.keys():
            if epoch not in all_results:
                all_results[epoch] = {}
            all_results[epoch][record["record_id"]] = record["results"]

rows = []
for epoch, epoch_results in all_results.items():
    label = EPOCH_TO_LABEL[epoch]
    for record_id, results in epoch_results.items():
        task_id, criterion_id = record_id.split("/", 1)
        valid_runs = [r for r in results if r is not None]
        all_agree = len(set(valid_runs)) <= 1 if valid_runs else None

        rows.append({
            "model": label,
            "epoch": epoch,
            "task_id": task_id,
            "criterion_id": criterion_id,
            "run_1": results[0] if len(results) > 0 else None,
            "run_2": results[1] if len(results) > 1 else None,
            "run_3": results[2] if len(results) > 2 else None,
            "all_agree": all_agree,
            "n_true": sum(1 for r in valid_runs if r is True),
            "n_false": sum(1 for r in valid_runs if r is False),
            "n_parse_fail": sum(1 for r in results if r is None),
        })

df = pd.DataFrame(rows)
df

,model,epoch,task_id,criterion_id,run_1,run_2,run_3,all_agree,n_true,n_false,n_parse_fail
0,Claude Sonnet 4.5 v2 prompt,1,1145,06a-04,False,False,False,True,0,3,0
1,Claude Sonnet 4.5 v2 prompt,1,711,06u1,True,True,True,True,3,0,0
2,Claude Sonnet 4.5 v2 prompt,1,137,137-06,True,True,True,True,3,0,0
3,Claude Sonnet 4.5 v2 prompt,1,137,03u2,False,False,False,True,0,3,0
4,Claude Sonnet 4.5 v2 prompt,1,711,09b-01,True,True,True,True,3,0,0
...,...,...,...,...,...,...,...,...,...,...,...
1211,Kimi K2 v4 prompt,16,106,106-03,True,True,True,True,3,0,0
1212,Kimi K2 v4 prompt,16,1145,01u3,True,True,True,True,3,0,0
1213,Kimi K2 v4 prompt,16,711,02u2,True,True,False,False,2,1,0
1214,Kimi K2 v4 prompt,16,711,711-05,True,True,True,True,3,0,0


In [3]:
# Summary stats per judge model
summary_rows = []
for epoch in all_results:
    label = EPOCH_TO_LABEL[epoch]
    model_df = df[df["epoch"] == epoch]
    n_criteria = len(model_df)
    
    n_true_run1 = model_df["run_1"].sum()
    n_true_run2 = model_df["run_2"].sum()
    n_true_run3 = model_df["run_3"].sum()
    
    stable_count = model_df["all_agree"].sum()
    stability_pct = stable_count / n_criteria * 100 if n_criteria else 0
    flip_pct = 100 - stability_pct
    
    parse_fails = model_df["n_parse_fail"].sum()
    
    summary_rows.append({
        "Judge Model": label,
        "# Criteria": n_criteria,
        "# True (run 1)": int(n_true_run1),
        "# True (run 2)": int(n_true_run2),
        "# True (run 3)": int(n_true_run3),
        "Stability (all 3 agree)": f"{stability_pct:.1f}%",
        "Flip rate (across 3 runs)": f"{flip_pct:.1f}%",
        "Parse failures": int(parse_fails),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

,Judge Model,# Criteria,# True (run 1),# True (run 2),# True (run 3),Stability (all 3 agree),Flip rate (across 3 runs),Parse failures
0,Claude Sonnet 4.5 v2 prompt,76,63,64,62,94.7%,5.3%,0
1,Claude Sonnet 4.6 v2 prompt,76,64,64,65,98.7%,1.3%,0
2,DeepSeek R1 v2 prompt,76,62,59,56,84.2%,15.8%,0
3,Kimi K2 v2 prompt,76,46,47,48,72.4%,27.6%,0
4,Claude Sonnet 4.5 v1 prompt,76,55,55,56,96.1%,3.9%,0
5,Claude Sonnet 4.6 v1 prompt,76,59,59,59,100.0%,0.0%,0
6,DeepSeek R1 v1 prompt,76,58,58,59,93.4%,6.6%,0
7,Kimi K2 v1 prompt,76,70,69,70,98.7%,1.3%,0
8,Claude Sonnet 4.5 v3 prompt,76,52,52,50,97.4%,2.6%,0
9,Claude Sonnet 4.6 v3 prompt,76,58,59,55,94.7%,5.3%,0


In [4]:
# Per-task stability breakdown
task_summary = df.groupby(["model", "task_id"]).agg(
    n_criteria=("all_agree", "count"),
    n_stable=("all_agree", "sum"),
).reset_index()
task_summary["stability"] = (task_summary["n_stable"] / task_summary["n_criteria"] * 100).round(1)
task_summary_pivot = task_summary.pivot_table(
    index="task_id", columns="model", values="stability"
)
task_summary_pivot

model,Claude Sonnet 4.5 v1 prompt,Claude Sonnet 4.5 v2 prompt,Claude Sonnet 4.5 v3 prompt,Claude Sonnet 4.5 v4 prompt,Claude Sonnet 4.6 v1 prompt,Claude Sonnet 4.6 v2 prompt,Claude Sonnet 4.6 v3 prompt,Claude Sonnet 4.6 v4 prompt,DeepSeek R1 v1 prompt,DeepSeek R1 v2 prompt,DeepSeek R1 v3 prompt,DeepSeek R1 v4 prompt,Kimi K2 v1 prompt,Kimi K2 v2 prompt,Kimi K2 v3 prompt,Kimi K2 v4 prompt
task_id,,,,,,,,,,,,,,,,
106,92.3,92.3,100.0,100.0,100.0,100.0,100.0,84.6,100.0,84.6,92.3,92.3,100.0,92.3,76.9,92.3
1145,100.0,92.9,100.0,92.9,100.0,100.0,92.9,92.9,100.0,78.6,100.0,92.9,100.0,71.4,71.4,100.0
137,90.5,95.2,95.2,100.0,100.0,100.0,100.0,100.0,90.5,90.5,100.0,95.2,100.0,47.6,76.2,85.7
145,100.0,92.9,100.0,100.0,100.0,92.9,78.6,100.0,85.7,92.9,85.7,85.7,100.0,92.9,71.4,85.7
711,100.0,100.0,92.9,100.0,100.0,100.0,100.0,92.9,92.9,71.4,85.7,78.6,92.9,71.4,64.3,78.6


In [5]:
# Flip analysis — criteria where the judgement changed across the 3 runs

flip_records = []
with open(results_path) as f:
    for line in f:
        record = json.loads(line)
        epoch = record["epoch"]
        if epoch not in EPOCH_TO_LABEL:
            continue
        results = record["results"]
        valid = [r for r in results if r is not None]
        if len(set(valid)) > 1:
            flip_records.append(record)

epoch_order = {epoch: i for i, epoch in enumerate(EPOCH_TO_LABEL)}
flip_records.sort(key=lambda r: epoch_order[r["epoch"]])

print(f"{len(flip_records)} flip cases across all models\n")

label_order = {label: i for i, label in enumerate(EPOCH_TO_LABEL.values())}
flip_summary = (
    pd.DataFrame([
        {"model": EPOCH_TO_LABEL[r["epoch"]], "record_id": r["record_id"]}
        for r in flip_records
    ])
    .groupby("model")
    .agg(n_flips=("record_id", "count"), record_ids=("record_id", lambda x: ", ".join(sorted(x))))
    .reset_index()
    .assign(order=lambda df: df["model"].map(label_order))
    .sort_values("order")
    .drop(columns="order")
)
display(flip_summary)

for record in flip_records:
    label = EPOCH_TO_LABEL[record["epoch"]]
    print(f"\n{'='*80}")
    print(f"Model : {label}  |  Record: {record['record_id']}  |  Results: {record['results']}")
    valid = [r for r in record["results"] if r is not None]
    majority = sum(valid) >= len(valid) / 2
    for i, (result, response) in enumerate(zip(record["results"], record["responses"])):
        excerpt = "\n    ".join((response or "").strip().splitlines()[-5:])
        marker = "  <<< MINORITY" if result != majority else ""
        print(f"\n  --- Run {i+1}: {result}{marker} ---")
        print(f"    {excerpt}")

101 flip cases across all models



,model,n_flips,record_ids
1,Claude Sonnet 4.5 v2 prompt,4,"106/08u2, 1145/06a-02, 137/05u1, 145/07a-01"
4,Claude Sonnet 4.6 v2 prompt,1,145/03u2
8,DeepSeek R1 v2 prompt,12,"106/02u2, 106/08u2, 1145/06a-02, 1145/06a-04, ..."
12,Kimi K2 v2 prompt,21,"106/02u2, 1145/04u1, 1145/05u1, 1145/06a-02, 1..."
0,Claude Sonnet 4.5 v1 prompt,3,"106/12b-01, 137/05u1, 137/137-03"
7,DeepSeek R1 v1 prompt,5,"137/137-02, 137/137-06, 145/07a-01, 145/07a-03..."
11,Kimi K2 v1 prompt,1,711/711-04
2,Claude Sonnet 4.5 v3 prompt,2,"137/137-02, 711/02u2"
5,Claude Sonnet 4.6 v3 prompt,4,"1145/06a-04, 145/05u1, 145/06u1, 145/07a-01"
9,DeepSeek R1 v3 prompt,5,"106/03u2, 145/02u2, 145/07a-01, 711/01u3, 711/..."



Model : Claude Sonnet 4.5 v2 prompt  |  Record: 106/08u2  |  Results: [False, True, True]

  --- Run 1: False  <<< MINORITY ---
    The AI response contains multiple suggestions that require resources explicitly stated as unavailable (projector/document camera) or highly unlikely to be available (laminating equipment, recording devices, regular printing capacity for take-home materials). While the response does include many appropriate low-resource strategies, the inclusion of these resource-dependent suggestions makes it partially inappropriate for the stated context.
    
    The criterion is asking whether the response is "not appropriate" - meaning it requires unavailable resources. Since the response does include suggestions requiring unavailable resources (projector, laminator, printing capacity, recording devices), it fails to be fully appropriate for the available resources.
    
    **false**

  --- Run 2: True ---
    The response demonstrates clear awareness of the resource

## Qualitative Analysis of Flip Reasoning

Claude Opus 4.6 (with medium thinking effort, inside Claude Code) analysed the reasoning traces of all flip cases (criteria where the 3 resampled runs did not unanimously agree) to categorise **why** the judge disagreed with itself. Each minority-vote reasoning trace was read and assigned to one of six failure-mode categories defined below.

| Code | Category | Description |
|---|---|---|
| **TD** | **Threshold disagreement** | Judges agree on the evidence but disagree on whether the issues are severe enough to cross the pass/fail boundary. E.g., "the response has minor resource-inappropriate suggestions but is 95% appropriate — does that pass?" |
| **CIA** | **Criterion interpretation ambiguity** | The criterion wording permits multiple valid readings. E.g., "accurate according to criteria" could mean matching the reference answer's specific framework, or being pedagogically sound in general. |
| **NPC** | **Negation/polarity confusion** | The criterion is phrased as a negative statement (e.g., "the response is NOT appropriate for…"), causing confusion about what `true`/`false` signifies. Judges agree on substance but flip the boolean. |
| **SMC** | **Scope of evaluation (meta-commentary)** | Disagreement about whether appended meta-text (e.g., "How does that land? Shall I adjust my approach?") should be evaluated as part of the in-character response. |
| **IER** | **Implicit vs. explicit requirement** | Disagreement about whether a response must explicitly address a factor (e.g., L1 script differences) or whether implicit accommodation (e.g., avoiding literacy-heavy tasks) suffices. |
| **ATC** | **Audience/target confusion** | Disagreement about who the response is written for (teacher vs. student; stakeholder vs. learner), affecting how to judge appropriateness of language level or content. |


Then Claude Opus 4.5 (with medium thinking effort, inside Claude Code) mapped each flip case to a category.

In [8]:
flip_categories = [
    "Claude Sonnet 4.5 v2 prompt==106/08u2==Minority judged partial inappropriate content as failure; majority accepted 95%+ appropriateness as passing=>TD",
    "Claude Sonnet 4.5 v2 prompt==137/05u1==Minority found core requirements met; majority weighted missing evening class/class size considerations as failure=>TD",
    "Claude Sonnet 4.5 v2 prompt==145/07a-01==Minority interpreted 'accurate' as matching reference framework; majority interpreted as being pedagogically sound=>CIA",
    "Claude Sonnet 4.5 v2 prompt==1145/06a-02==Majority focused on substantive response being immediate; minority judged meta-commentary as breaking real-time simulation=>SMC",
    "Claude Sonnet 4.6 v2 prompt==145/03u2==Minority accepted implicit appropriateness; majority required explicit addressing of L1 script/literacy=>IER",
    "DeepSeek R1 v2 prompt==1145/06a-02==Majority accepted core response as immediate; minority judged meta-commentary breaks real-time flow=>SMC",
    "DeepSeek R1 v2 prompt==137/06u1==Minority interpreted 'appropriate for' as requiring proactive engagement; majority accepted avoidance as appropriate=>CIA",
    "DeepSeek R1 v2 prompt==106/08u2==Minority said false but reasoned response IS appropriate; classic polarity error on criterion framing=>NPC",
    "DeepSeek R1 v2 prompt==711/711-04==Minority accepted implicit identification via averages; majority required explicit student-6 highlighting=>IER",
    "DeepSeek R1 v2 prompt==145/07a-01==Minority required actual evaluation not just framework; majority accepted framework accuracy=>CIA",
    "DeepSeek R1 v2 prompt==106/02u2==Minority judged AI's complex language inappropriate; majority understood criterion refers to student-facing language=>ATC",
    "DeepSeek R1 v2 prompt==137/137-02==Minority accepted approach keeps class together; majority found segregation in specific suggestions=>TD",
    "DeepSeek R1 v2 prompt==1145/06u2==Runs 2 and 3 reasoned response IS appropriate then output false; polarity error on negatively framed criterion=>NPC",
    "DeepSeek R1 v2 prompt==711/711-07==Minority found programme design considerations; majority found analysis attributes to skill difficulty not design=>TD",
    "DeepSeek R1 v2 prompt==711/711-01==Minority required granular student-by-student evidence; majority accepted aggregate claims=>IER",
    "DeepSeek R1 v2 prompt==711/06u1==Run 1 answered false meaning appropriate; majority answered true meaning appropriate; same conclusion different boolean=>NPC",
    "DeepSeek R1 v2 prompt==1145/06a-04==Minority judged stays in character; majority found meta-commentary violates criterion=>SMC",
    "Kimi K2 v2 prompt==1145/06a-02==Minority found response overly lengthy for real-time; majority accepted simulation as appropriate=>TD",
    "Kimi K2 v2 prompt==137/02a-05==Minority found balance achieved; majority judged response form-heavy=>TD",
    "Kimi K2 v2 prompt==137/137-05==Minority found scaffolding appropriate without stigma; majority found explicit labeling creates stigma=>TD",
    "Kimi K2 v2 prompt==1145/06a-04==Majority accepted brief facilitation check as acceptable; minority found meta-commentary breaks role=>SMC",
    "Kimi K2 v2 prompt==711/711-05==Minority focused on data errors invalidating presentation; majority evaluated format appropriateness only=>CIA",
    "Kimi K2 v2 prompt==137/02u2==Minority judged language appropriate for context; majority found language exceeds A2 level=>TD",
    "Kimi K2 v2 prompt==1145/04u1==Minority found AI dominates reducing learning value; majority found it serves learning purpose=>TD",
    "Kimi K2 v2 prompt==137/02a-06==Minority found metacognition neglected; majority found 5-minute reflection meets minimum=>TD",
    "Kimi K2 v2 prompt==711/06u1==Minority reinterpreted criterion polarity incorrectly; majority correctly judged cultural appropriateness=>NPC",
    "Kimi K2 v2 prompt==1145/05u1==Minority found meta-commentary inappropriate for context; majority accepted overall appropriateness=>SMC",
    "Kimi K2 v2 prompt==137/05u1==Minority found core requirements met despite gaps; majority required German L1 and evening context=>TD",
    "Kimi K2 v2 prompt==711/02u2==Minority focused on data errors; majority evaluated language appropriateness separately=>CIA",
    "Kimi K2 v2 prompt==137/02a-08==Minority found clear scaffolding progression; majority found activities remain bifurcated=>TD",
    "Kimi K2 v2 prompt==137/02a-02==Minority found progression issues for weaker students; majority found clear skill building=>TD",
    "Kimi K2 v2 prompt==137/137-02==Minority found some division in recommendations; majority accepted differentiation within shared activities=>TD",
    "Kimi K2 v2 prompt==137/02a-09==Minority found activities engaging; majority found separated pairings reduce engagement=>TD",
    "Kimi K2 v2 prompt==145/07a-01==Minority found B2 descriptor inaccuracies; majority found framework accurate overall=>TD",
    "Kimi K2 v2 prompt==137/02a-03==Minority found pedagogical issues with mixed-ability management; majority found approach appropriate=>TD",
    "Kimi K2 v2 prompt==137/137-03==Minority found adaptations require more prep than stated; majority found realistic with alternatives=>TD",
    "Kimi K2 v2 prompt==711/01u3==Minority focused on data analysis errors; majority evaluated age-appropriateness of framing=>CIA",
    "Kimi K2 v2 prompt==106/02u2==Minority found A1 activities too complex; majority found language appropriate for young learners=>ATC",
    "Claude Sonnet 4.5 v1 prompt==137/05u1==Minority accepted general appropriateness; majority required more contextual depth=>TD",
    "Claude Sonnet 4.5 v1 prompt==137/137-03==Minority found some suggestions prep-intensive; majority found realistic overall=>TD",
    "Claude Sonnet 4.5 v1 prompt==106/12b-01==Minority found appropriate for professional level; majority found impractical for under-resourced teacher=>TD",
    "DeepSeek R1 v1 prompt==137/137-06==Minority gave true; majority gave false; threshold on timing feasibility=>TD",
    "DeepSeek R1 v1 prompt==711/09b-02==Minority gave true; majority gave false; interpretation of recognised standards=>CIA",
    "DeepSeek R1 v1 prompt==137/137-02==Minority gave false; majority gave true; threshold on keeping class together=>TD",
    "DeepSeek R1 v1 prompt==145/07a-01==Minority gave true; majority gave false; interpretation of evaluation accuracy=>CIA",
    "DeepSeek R1 v1 prompt==145/07a-03==Minority gave false; majority gave true; interpretation of explicit criteria=>CIA",
    "Kimi K2 v1 prompt==711/711-04==Minority gave false; majority gave true; implicit vs explicit student identification=>IER",
    "Claude Sonnet 4.5 v3 prompt==711/02u2==Minority questioned which CEFR level applies; majority correctly identified stakeholder audience=>ATC",
    "Claude Sonnet 4.5 v3 prompt==137/137-02==Minority found speaking activity divides room; majority found differentiation within shared activities=>TD",
    "Claude Sonnet 4.6 v3 prompt==145/05u1==Minority weighted NZ context omissions heavily; majority found core elements addressed=>TD",
    "Claude Sonnet 4.6 v3 prompt==145/07a-01==Minority required actual student evaluation; majority accepted framework accuracy=>CIA",
    "Claude Sonnet 4.6 v3 prompt==145/06u1==Minority found eye contact minor issue; majority agreed with only minor concern=>TD",
    "Claude Sonnet 4.6 v3 prompt==1145/06a-04==Minority accepted facilitation check as acceptable; majority found it breaks character=>SMC",
    "DeepSeek R1 v3 prompt==106/03u2==Minority found implicit accommodation of L1 script; majority required explicit addressing=>IER",
    "DeepSeek R1 v3 prompt==145/07a-01==Minority required actual evaluation performed; majority accepted framework as accurate=>CIA",
    "DeepSeek R1 v3 prompt==711/01u3==Minority found response not for learners but stakeholders; majority found appropriate about learners=>ATC",
    "DeepSeek R1 v3 prompt==711/02u2==Minority judged language too complex for B2 learners; majority identified stakeholders as audience=>ATC",
    "DeepSeek R1 v3 prompt==145/02u2==Minority found language exceeds B2; majority found appropriate for teacher audience=>ATC",
    "Kimi K2 v3 prompt==1145/06u2==Minority gave false without clear reasoning; majority found culturally appropriate=>NPC",
    "Kimi K2 v3 prompt==711/05u1==Minority found strong contextual awareness; majority found missing Chinese educational culture depth=>TD",
    "Kimi K2 v3 prompt==1145/04u1==Minority found response inappropriate for learning purpose; majority found it serves learning goal=>TD",
    "Kimi K2 v3 prompt==145/07a-01==Minority required actual evaluation not framework; majority accepted framework accuracy=>CIA",
    "Kimi K2 v3 prompt==711/711-01==Minority found claim verified by data; majority required explicit student-level demonstration=>IER",
    "Kimi K2 v3 prompt==711/06u1==Minority misinterpreted criterion polarity; majority correctly judged cultural appropriateness=>NPC",
    "Kimi K2 v3 prompt==145/06u1==Minority found framework culturally neutral; majority found eye contact assumption problematic=>TD",
    "Kimi K2 v3 prompt==137/05u1==Minority found German L1 not addressed; majority found contextual factors comprehensively addressed=>TD",
    "Kimi K2 v3 prompt==137/02a-05==Minority found balance achieved; majority found form-heavy approach=>TD",
    "Kimi K2 v3 prompt==137/137-05==Minority found station options create separation; majority found scaffolding avoids stigma=>TD",
    "Kimi K2 v3 prompt==106/12b-01==Minority found response not calibrated for novice teacher; majority found practical guidance appropriate=>TD",
    "Kimi K2 v3 prompt==145/07a-03==Minority found meaning-first fallback not applicable; majority found clear criteria provided=>CIA",
    "Kimi K2 v3 prompt==711/09b-02==Minority found CEFR reference sufficient; majority found analysis lacks external standards=>TD",
    "Kimi K2 v3 prompt==106/06u1==Minority found culturally neutral acceptable; majority found missing Ethiopian cultural specifics=>CIA",
    "Kimi K2 v3 prompt==137/02u2==Minority found criterion asks for B2 student language; majority found teacher-appropriate=>ATC",
    "Kimi K2 v3 prompt==1145/06a-02==Minority found response overly monologic; majority found real-time simulation successful=>TD",
    "Kimi K2 v3 prompt==137/137-06==Minority found timing feasible with discipline; majority found adaptations add time pressure=>TD",
    "Kimi K2 v3 prompt==106/08u2==Minority found document camera suggestion inappropriate; majority found core strategies resource-free=>TD",
    "Kimi K2 v3 prompt==711/01u3==Minority found response not for learner age group; majority found appropriate about adult learners=>ATC",
    "Kimi K2 v3 prompt==145/01u2==Minority found framework too complex for age group; majority found expectations appropriate=>TD",
    "Kimi K2 v3 prompt==1145/05u1==Minority found meta-commentary and stage directions inappropriate; majority found overall context appropriate=>SMC",
    "Claude Sonnet 4.5 v4 prompt==1145/06a-02==Minority accepted substantive response as real-time; majority found meta-commentary disrupts simulation=>SMC",
    "Claude Sonnet 4.6 v4 prompt==106/03u2==Minority accepted implicit L1 accommodation; majority required explicit addressing=>IER",
    "Claude Sonnet 4.6 v4 prompt==711/711-07==Minority confused criterion polarity on programme design; majority correctly found design factors considered=>NPC",
    "Claude Sonnet 4.6 v4 prompt==106/08u2==Minority confused criterion polarity framing; majority correctly judged response appropriate=>NPC",
    "Claude Sonnet 4.6 v4 prompt==1145/06a-04==Minority accepted closing as facilitative not explanatory; majority found meta-comment violates criterion=>SMC",
    "DeepSeek R1 v4 prompt==711/06u1==Minority judged avoiding sensitive topics as true; majority judged same conclusion as false=>NPC",
    "DeepSeek R1 v4 prompt==711/09b-02==Minority found performance tiers and CEFR references sufficient; majority required external benchmarks=>CIA",
    "DeepSeek R1 v4 prompt==106/03u2==Minority found implicit prioritization of oral skills addresses L1 script; majority required explicit mention=>IER",
    "DeepSeek R1 v4 prompt==1145/05u1==Minority found response appropriate for one-on-one video call; majority found no class context consideration=>CIA",
    "DeepSeek R1 v4 prompt==145/07a-01==Minority required actual student evaluation performed; majority accepted framework accuracy=>CIA",
    "DeepSeek R1 v4 prompt==145/06u1==Minority found criterion met by avoidance; majority found response doesn't engage with sensitivity areas=>CIA",
    "DeepSeek R1 v4 prompt==711/711-07==Minority found programme design factors considered; majority found same factors present=>NPC",
    "DeepSeek R1 v4 prompt==137/03u2==Minority required explicit L1 mention; majority found pedagogical strategies implicitly appropriate=>IER",
    "Kimi K2 v4 prompt==711/04u1==Minority found calculation errors invalidate purpose; majority found framing and interpretation appropriate=>CIA",
    "Kimi K2 v4 prompt==145/07a-01==Minority found framework internally consistent; majority found no evaluation performed to judge=>CIA",
    "Kimi K2 v4 prompt==711/711-07==Minority misinterpreted criterion checking; majority correctly found programme design considered=>NPC",
    "Kimi K2 v4 prompt==137/02a-05==Minority found response form-focused; majority found balance achieved=>TD",
    "Kimi K2 v4 prompt==145/07a-03==Minority found meaning-first principle absent; majority found clear criteria eliminate need for fallback=>CIA",
    "Kimi K2 v4 prompt==106/08u2==Minority found partial inappropriate suggestions; majority found core strategies resource-appropriate=>TD",
    "Kimi K2 v4 prompt==137/137-06==Minority found timing feasible if disciplined; majority found complexity exceeds time allocation=>TD",
    "Kimi K2 v4 prompt==137/03u2==Minority found L1 omission critical; majority found implicit appropriateness for German adults=>IER",
    "Kimi K2 v4 prompt==711/02u2==Minority questioned B2 vs stakeholder audience; majority identified professional register appropriate=>ATC",
]

In [9]:
# Verify flip_categories matches flip_records
flip_records_set = {(EPOCH_TO_LABEL[r["epoch"]], r["record_id"]) for r in flip_records}
flip_categories_set = set()
for entry in flip_categories:
    last_sep = entry.rfind("=>")
    rest = entry[:last_sep]
    parts = rest.split("==", 2)
    flip_categories_set.add((parts[0], parts[1]))

print(f"flip_records: {len(flip_records_set)} cases")
print(f"flip_categories: {len(flip_categories_set)} cases")

missing_from_categories = flip_records_set - flip_categories_set
extra_in_categories = flip_categories_set - flip_records_set

if missing_from_categories:
    print(f"\n❌ Missing from flip_categories ({len(missing_from_categories)}):")
    for model, rid in sorted(missing_from_categories):
        print(f"  {model} == {rid}")

if extra_in_categories:
    print(f"\n❌ Extra in flip_categories ({len(extra_in_categories)}):")
    for model, rid in sorted(extra_in_categories):
        print(f"  {model} == {rid}")

if not missing_from_categories and not extra_in_categories:
    print("\n✓ flip_categories is consistent with flip_records")

flip_records: 101 cases
flip_categories: 101 cases

✓ flip_categories is consistent with flip_records


In [10]:

# Parse flip_categories into a DataFrame
parsed = []
for entry in flip_categories:
    last_sep = entry.rfind("=>")
    category = entry[last_sep + 2:]
    rest = entry[:last_sep]
    parts = rest.split("==", 2)
    model_label = parts[0]
    record_id = parts[1]
    base_model = model_label.rsplit(" v", 1)[0]
    parsed.append({"model_label": model_label, "base_model": base_model, "record_id": record_id, "category": category})

flip_cat_df = pd.DataFrame(parsed)

CATEGORY_NAMES = {
    "TD":  "Threshold disagreement",
    "CIA": "Criterion interpretation ambiguity",
    "NPC": "Negation/polarity confusion",
    "SMC": "Scope of evaluation (meta-commentary)",
    "IER": "Implicit vs. explicit requirement",
    "ATC": "Audience/target confusion",
}
CAT_ORDER = list(CATEGORY_NAMES.keys())

# --- Overall distribution ---
counts = flip_cat_df["category"].value_counts().reindex(CAT_ORDER, fill_value=0)
overall_df = pd.DataFrame({
    "Code": CAT_ORDER,
    "Category": [CATEGORY_NAMES[c] for c in CAT_ORDER],
    "Count": counts.values,
    "% of flips": (counts.values / len(flip_cat_df) * 100).round(1),
})
print(f"Overall failure-mode distribution across {len(flip_cat_df)} flip cases\n")
display(overall_df.to_string(index=False))

# --- Breakdown by base model ---
pivot = (
    flip_cat_df
    .groupby(["base_model", "category"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=CAT_ORDER, fill_value=0)
)
print("\nFlip-case counts by base model and failure-mode category\n")
display(pivot)

# --- Breakdown by prompt version ---
def extract_version(label):
    try:
        return "v" + label.rsplit(" v", 1)[1].split(" ")[0]
    except Exception:
        return "unknown"

flip_cat_df["prompt_version"] = flip_cat_df["model_label"].apply(extract_version)
version_pivot = (
    flip_cat_df
    .groupby(["prompt_version", "category"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=CAT_ORDER, fill_value=0)
)
print("\nFlip-case counts by scorer prompt version and failure-mode category\n")
display(version_pivot)


Overall failure-mode distribution across 101 flip cases



'Code                              Category  Count  % of flips\n  TD                Threshold disagreement     41        40.6\n CIA    Criterion interpretation ambiguity     21        20.8\n NPC           Negation/polarity confusion     11        10.9\n SMC Scope of evaluation (meta-commentary)      9         8.9\n IER     Implicit vs. explicit requirement     10         9.9\n ATC             Audience/target confusion      9         8.9'


Flip-case counts by base model and failure-mode category



category,TD,CIA,NPC,SMC,IER,ATC
base_model,,,,,,
Claude Sonnet 4.5,6,1,0,2,0,1
Claude Sonnet 4.6,2,1,2,2,2,0
DeepSeek R1,4,10,5,2,5,4
Kimi K2,29,9,4,3,3,4



Flip-case counts by scorer prompt version and failure-mode category



category,TD,CIA,NPC,SMC,IER,ATC
prompt_version,,,,,,
v1,5,3,0,0,1,0
v2,18,6,4,5,3,2
v3,15,5,2,2,2,6
v4,3,7,5,2,4,1


## Qualitative Examples of Judge Instability by Failure-Mode Category

The examples below are drawn from the 101 flip cases — criteria where three independent resampled judge runs did not unanimously agree. For each category one representative case is shown, followed by a brief interpretation of why the disagreement arose.

---

### TD — Threshold disagreement (41 cases, 40.6%)

**Case:** `Claude Sonnet 4.5 v2 prompt`, criterion `106/08u2`

> *Minority judged partial inappropriate content as failure; majority accepted 95%+ appropriateness as passing.*

Both majority and minority judges identified the same evidence: the response contained mostly resource-appropriate suggestions with a small number of items that assumed equipment availability. The disagreement was purely about where to draw the line — whether 95% appropriateness clears the bar or whether any inappropriate suggestions constitute failure. This is a boundary-precision problem where the criterion's threshold sits exactly at the point the response occupies.

---

### CIA — Criterion interpretation ambiguity (21 cases, 20.8%)

**Case:** `Claude Sonnet 4.5 v2 prompt`, criterion `145/07a-01`

> *Minority interpreted 'accurate' as matching reference framework; majority interpreted as being pedagogically sound.*

The criterion phrase "accurate according to the criteria or mark key" admits two grammatically valid readings: (a) accurate relative to a supplied marking key, or (b) accurate in general pedagogical terms. The minority judge looked for alignment with the reference answer's specific CEFR descriptors; the majority judges evaluated whether the response contained pedagogically sound principles generally. This ambiguity persisted across all four prompt versions and multiple models for this criterion.

---

### NPC — Negation/polarity confusion (11 cases, 10.9%)

**Case:** `DeepSeek R1 v2 prompt`, criterion `1145/06u2`

> *Runs 2 and 3 reasoned response IS appropriate then output false; polarity error on negatively framed criterion.*

The criterion tests whether the response is "NOT appropriate for cultural sensitivities" — a negative framing where `true` means failure. Runs 2 and 3 correctly concluded the response was culturally appropriate, but then outputted `false` instead of `true`, inverting the boolean. This is a pure surface-form parsing error that produces a wrong label despite correct substantive reasoning. The pattern appeared repeatedly with DeepSeek R1 on negatively-framed criteria.

---

### SMC — Scope of evaluation / meta-commentary (9 cases, 8.9%)

**Case:** `Kimi K2 v2 prompt`, criterion `1145/06a-04`

> *Majority accepted brief facilitation check as acceptable; minority found meta-commentary breaks role.*

The AI response consisted of an in-character supplier roleplay followed by a brief out-of-character facilitation check ("How does that land?"). The criterion tested whether the AI "stays in character." Majority judges scoped evaluation to the substantive roleplay content and passed; the minority judge treated the appended meta-text as part of the response and failed it for breaking character. Neither interpretation is inherently wrong — the criterion does not specify whether supplementary framing text is in scope.

---

### IER — Implicit vs. explicit requirement (10 cases, 9.9%)

**Case:** `DeepSeek R1 v3 prompt`, criterion `106/03u2`

> *Minority found implicit accommodation of L1 script; majority required explicit addressing.*

The criterion required that the response be "appropriate for learners' L1 backgrounds" (Ethiopian students with Ge'ez script). The AI response never explicitly mentioned Ethiopic script or L1 literacy but consistently avoided reading/writing tasks and centred phonemic-oral activities. The minority judge passed it because the pedagogical choices implicitly accommodate these characteristics; the majority judges required explicit acknowledgement. The criterion wording does not settle this: "appropriate for" could mean "explicitly addresses" or "does not harm."

---

### ATC — Audience/target confusion (9 cases, 8.9%)

**Case:** `DeepSeek R1 v3 prompt`, criterion `711/02u2`

> *Minority judged language too complex for B2 learners; majority identified stakeholders as audience.*

The AI was asked to write a report for an engineering department about B2-level students' performance. The criterion tested whether the language was "appropriate for the CEFR level." The minority judge read this as requiring the report itself to be written in B2-accessible prose and failed it for C1-C2 register; the majority judges correctly identified that the report's audience was academic administrators, not the students themselves. This pattern recurred across multiple models on criteria involving professional reports about language learners.